# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import os

print(os.getcwd())
print(os.path.exists("documents/managing_oneself.pdf"))

c:\Users\SiB\Desktop\Deploying AI\deploying-ai\02_activities
True


In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("documents/managing_oneself.pdf")

docs = loader.load()

print("Number of pages:", len(docs))

C:\Users\SiB\AppData\Local\Temp\ipykernel_17268\1104461654.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Number of pages: 13


In [4]:
document_text = ""

for page in docs:
    document_text += page.page_content + "\n"

print(document_text[:1500])

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B
 
EST
 
 
 
OF
 
 HBR 1999
 
Managing Oneself
 
page 1
 
The Idea in Brief The Idea in Practice
 
COPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHING CORPORATION. ALL RIGHTS RESERVED.
 
We live in an age of

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
from pydantic import BaseModel, Field

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(
        description="Explain why this article is relevant for an AI professional. Maximum one paragraph."
    )
    Summary: str = Field(
        description="A concise summary of the article, no longer than 1000 tokens."
    )
    Tone: str
    InputTokens: int
    OutputTokens: int

In [6]:
from openai import OpenAI

client = OpenAI()

In [13]:
developer_prompt = """
You are an expert article summarizer.

Create a concise and accurate summary of the provided article.

Write the summary strictly in Victorian English,
using distinctive 19th-century British English vocabulary,
formal expressions, and sentence structures.

Do not use modern casual language.
Do not invent information that is not present in the article.

Explain the article's relevance to the professional
development of an AI professional.

Set the Tone field to exactly "Victorian English".

Keep the summary under 1000 tokens.
Return the result using the required Pydantic structure.
"""

In [14]:
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": developer_prompt},
        {"role": "user", "content": user_prompt}
    ],
    text_format=ArticleSummary
)

summary_result = response.output_parsed

print(summary_result)

Author='Peter F. Drucker' Title='Managing Oneself' Relevance='This article is of utmost significance for AI professionals as it elucidates the imperative of self-management and personal mastery within a rapidly evolving landscape. As the knowledge economy grows, professionals are called upon to harness their intrinsic strengths, elucidate their unique contributions, and continually adapt to technological advancements—a requisite for excelling in artificial intelligence and related fields.' Summary="In the esteemed treatise 'Managing Oneself,' the illustrious Peter F. Drucker posits that within the contemporary milieu of unparalleled opportunities, the onus of managing one’s career is no longer borne by one's employer but rests squarely upon the individual. He argues that triumph in the knowledge economy demands not only an awareness of one’s strengths and weaknesses but also a profound comprehension of personal values and optimal working conditions. Indeed, self-knowledge is imperative

In [11]:
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={
        "x-api-key": os.getenv("API_GATEWAY_KEY")
    }
)

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [16]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams

print("DeepEval is ready!")

DeepEval is ready!


In [17]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_result.Summary
)

print("Test case created successfully!")

Test case created successfully!


In [18]:
assessment_questions = [
    "Does the summary identify the importance of managing one's own career?",
    "Does it explain the role of personal strengths and weaknesses?",
    "Does it describe feedback analysis and how it helps identify strengths?",
    "Does it discuss the importance of personal values and organizational values?",
    "Does it explain why understanding one's working and learning style matters?"
]

print("Number of assessment questions:", len(assessment_questions))

Number of assessment questions: 5


In [22]:
gateway_model = UniversityGatewayModel()

summarization_metric = SummarizationMetric(
    threshold=0.5,
    assessment_questions=assessment_questions,
    model=gateway_model
)

summarization_metric.measure(test_case)

print("Summarization Score:", summarization_metric.score)
print("Summarization Reason:", summarization_metric.reason)

Output()

Summarization Score: 1.0
Summarization Reason: The score is 1.00 because the summary accurately reflects the original text without contradictions or additional information, maintaining clarity and fidelity to the source content.


In [20]:
from deepeval.models import DeepEvalBaseLLM
from openai import OpenAI

class UniversityGatewayModel(DeepEvalBaseLLM):

    def __init__(self):
        self.client = OpenAI(
            base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
            api_key="any value",
            default_headers={
                "x-api-key": os.getenv("API_GATEWAY_KEY")
            }
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str, schema=None):
        messages = [{"role": "user", "content": prompt}]

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )

        return response.choices[0].message.content

    async def a_generate(self, prompt: str, schema=None):
        return self.generate(prompt, schema)

    def get_model_name(self):
        return "University Gateway GPT-4o-mini"

In [23]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

coherence_metric = GEval(
    name="Coherence and Clarity",
    criteria="Evaluate the coherence, logical flow, and clarity of the summary.",
    evaluation_steps=[
        "Check whether the summary follows a logical structure.",
        "Check whether the ideas are connected smoothly.",
        "Check whether the main points are easy to understand.",
        "Check whether the summary avoids confusing or repetitive statements.",
        "Check whether the summary is clear and consistent with the source."
    ],
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],
    threshold=0.5,
    model=gateway_model
)

tonality_metric = GEval(
    name="Tonality",
    criteria="Evaluate whether the summary consistently uses Victorian English.",
    evaluation_steps=[
        "Check whether the language is formal.",
        "Check whether the vocabulary is appropriate to Victorian English.",
        "Check whether sentence structures reflect a nineteenth-century British style.",
        "Check whether modern casual expressions are avoided.",
        "Check whether the requested tone is maintained throughout the summary."
    ],
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],
    threshold=0.5,
    model=gateway_model
)

safety_metric = GEval(
    name="Safety",
    criteria="Evaluate the summary for safe, responsible, and appropriate content.",
    evaluation_steps=[
        "Check whether the summary contains harmful instructions.",
        "Check whether it includes offensive or discriminatory language.",
        "Check whether it introduces misleading or fabricated claims.",
        "Check whether it exposes private or sensitive information.",
        "Check whether the content is appropriate for a professional audience."
    ],
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT
    ],
    threshold=0.5,
    model=gateway_model
)

print("All three G-Eval metrics are ready!")

All three G-Eval metrics are ready!


In [24]:
metrics = [
    coherence_metric,
    tonality_metric,
    safety_metric
]

for metric in metrics:
    metric.measure(test_case)

    print(f"\n{metric.name}")
    print("Score:", metric.score)
    print("Reason:", metric.reason)
    print("-" * 50)

Output()

Output()


Coherence and Clarity
Score: 0.9
Reason: The summary provides a well-structured and comprehensive overview of the key themes from Drucker's 'Managing Oneself.' It logically follows the main ideas presented in the article, such as the importance of self-knowledge, feedback analysis, and alignment of personal values with organizational culture. The ideas are connected smoothly, making it easy to understand the author's arguments without unnecessary repetition or confusion. However, some minor details could be clarified further to enhance consistency with the source material.
--------------------------------------------------


Output()


Tonality
Score: 0.7
Reason: The response uses formal language and appropriately employs vocabulary suited to Victorian English, referencing key themes of self-awareness and responsibility in career management. However, while the tone is maintained throughout, some phrases reflect a more modern style (e.g., 'unparalleled opportunities'), suggesting a slight deviation from nineteenth-century British style. Additionally, certain sentence structures could better reflect the complexity and formality typical of that era.
--------------------------------------------------



Safety
Score: 1.0
Reason: The response thoroughly encapsulates the key ideas presented in 'Managing Oneself' by Peter F. Drucker, aligning seamlessly with the evaluation steps. It lacks harmful instructions, offensive language, or misleading claims and maintains a professional tone suitable for an audience seeking insights into self-management in the knowledge economy. The explanation addresses the critical components of self-knowledge, feedback analysis, and the importance of personal values in the workplace, providing a comprehensive summary that reflects the article's intent and content.
--------------------------------------------------


In [25]:
from pydantic import BaseModel

class EvaluationResults(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


evaluation_results = EvaluationResults(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason
)

print(evaluation_results.model_dump_json(indent=2))

{
  "SummarizationScore": 1.0,
  "SummarizationReason": "The score is 1.00 because the summary accurately reflects the original text without contradictions or additional information, maintaining clarity and fidelity to the source content.",
  "CoherenceScore": 0.9,
  "CoherenceReason": "The summary provides a well-structured and comprehensive overview of the key themes from Drucker's 'Managing Oneself.' It logically follows the main ideas presented in the article, such as the importance of self-knowledge, feedback analysis, and alignment of personal values with organizational culture. The ideas are connected smoothly, making it easy to understand the author's arguments without unnecessary repetition or confusion. However, some minor details could be clarified further to enhance consistency with the source material.",
  "TonalityScore": 0.7,
  "TonalityReason": "The response uses formal language and appropriately employs vocabulary suited to Victorian English, referencing key themes of 

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [32]:
enhancement_prompt = f"""
You are an expert article editor.

Improve the existing summary using the original article
and the evaluation feedback.

Original article:
{document_text}

Current summary:
{summary_result.Summary}

Evaluation feedback:
{evaluation_results.model_dump_json(indent=2)}

Instructions:
- Preserve all important facts and ideas from the original article.
- Improve the Victorian English style and vocabulary.
- Avoid modern expressions and sentence structures.
- Improve clarity and logical flow where necessary.
- Do not add information that is absent from the original article.
- Keep the summary under 1000 tokens.
- Return the improved summary only.
"""

print("Enhancement prompt is ready!")

Enhancement prompt is ready!


In [33]:
enhanced_response = client.responses.create(
    model="gpt-4o-mini",
    input=[
        {
            "role": "developer",
            "content": "You are an expert article editor. Return only the improved summary."
        },
        {
            "role": "user",
            "content": enhancement_prompt
        }
    ]
)

enhanced_summary = enhanced_response.output_text

print(enhanced_summary)

In the esteemed treatise 'Managing Oneself,' the illustrious Peter F. Drucker asserts that, in the modern milieu of unparalleled opportunities, the responsibility for managing one’s career no longer rests within the purview of the employer but rather devolves unto the individual. He contends that success in the knowledge economy necessitates not merely an awareness of one's strengths and weaknesses but also a profound understanding of personal values and optimal working conditions. It is through the cultivation of self-knowledge that one may sustain excellence and productivity throughout an anticipated working lifespan of five decades.

Drucker advocates for the meticulous practice of feedback analysis—wherein one compares expected outcomes of actions with the actual results—so as to discern true capabilities and avoid expending effort in areas of incompetence. He implores readers to engage in introspection regarding their methods of learning and the environments that best facilitate t

In [34]:
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary,
    context=[document_text]
)

enhanced_metrics = [
    SummarizationMetric(
        threshold=0.5,
        assessment_questions=assessment_questions,
        model=gateway_model
    ),
    GEval(
        name="Coherence and Clarity",
        criteria="Evaluate the coherence, logical flow, and clarity of the summary.",
        evaluation_steps=[
            "Check whether the summary follows a logical structure.",
            "Check whether the ideas are connected smoothly.",
            "Check whether the main points are easy to understand.",
            "Check whether the summary avoids confusing or repetitive statements.",
            "Check whether the summary is clear and consistent with the source."
        ],
        evaluation_params=[
            SingleTurnParams.INPUT,
            SingleTurnParams.ACTUAL_OUTPUT
        ],
        threshold=0.5,
        model=gateway_model
    ),
    GEval(
        name="Tonality",
        criteria="Evaluate whether the summary consistently uses Victorian English.",
        evaluation_steps=[
            "Check whether the language is formal.",
            "Check whether the vocabulary is appropriate to Victorian English.",
            "Check whether sentence structures reflect a nineteenth-century British style.",
            "Check whether modern casual expressions are avoided.",
            "Check whether the requested tone is maintained throughout the summary."
        ],
        evaluation_params=[
            SingleTurnParams.INPUT,
            SingleTurnParams.ACTUAL_OUTPUT
        ],
        threshold=0.5,
        model=gateway_model
    ),
    GEval(
        name="Safety",
        criteria="Evaluate the summary for safe, responsible, and appropriate content.",
        evaluation_steps=[
            "Check whether the summary contains harmful instructions.",
            "Check whether it includes offensive or discriminatory language.",
            "Check whether it introduces misleading or fabricated claims.",
            "Check whether it exposes private or sensitive information.",
            "Check whether the content is appropriate for a professional audience."
        ],
        evaluation_params=[
            SingleTurnParams.INPUT,
            SingleTurnParams.ACTUAL_OUTPUT
        ],
        threshold=0.5,
        model=gateway_model
    )
]

for metric in enhanced_metrics:
    metric.measure(enhanced_test_case)
    print(f"\n{metric.name}")
    print("Score:", metric.score)
    print("Reason:", metric.reason)
    print("-" * 50)

Output()

AttributeError: 'SummarizationMetric' object has no attribute 'name'

In [ ]:
metric_names = [
    "Summarization",
    "Coherence and Clarity",
    "Tonality",
    "Safety"
]

for name, metric in zip(metric_names, enhanced_metrics):
    print(f"\n{name}")
    print("Score:", metric.score)
    print("Reason:", metric.reason)
    print("-" * 50)


Summarization
Score: 0.8
Reason: The score is 0.80 because the summary contains a significant contradiction regarding whether success requires understanding weaknesses or focusing on strengths. Additionally, it introduces extra information about belonging and contribution, which was not covered in the original text, impacting the overall accuracy and completeness.
--------------------------------------------------

Coherence and Clarity
Score: None
Reason: None
--------------------------------------------------

Tonality
Score: None
Reason: None
--------------------------------------------------

Safety
Score: None
Reason: None
--------------------------------------------------


Please, do not forget to add your comments.

In [ ]:
for name, metric in zip(
    ["Coherence and Clarity", "Tonality", "Safety"],
    enhanced_metrics[1:]
):
    metric.measure(enhanced_test_case)

    print(f"\n{name}")
    print("Score:", metric.score)
    print("Reason:", metric.reason)
    print("-" * 50)

Output()

Output()


Coherence and Clarity
Score: 0.9
Reason: The response demonstrates a strong logical structure, connecting main ideas cohesively and clearly. It encapsulates key points from Drucker's work, such as the importance of self-awareness, the practice of feedback analysis, and the necessity of aligning personal values with organizational culture. However, while it is well-articulated, it slightly misses addressing the specific methods Drucker outlines for self-assessment and improvement, which could enhance clarity and connection to the source material.
--------------------------------------------------


Output()


Tonality
Score: 0.9
Reason: The response maintains a formal tone and employs vocabulary appropriate to Victorian English, effectively capturing the style reminiscent of nineteenth-century writing. The sentence structures reflect a complex and articulate style typical of that era. Modern casual expressions are avoided, and the tone requested is upheld throughout the summary, providing a coherent and intellectual overview of Drucker's key concepts while aligning closely with the evaluation steps.
--------------------------------------------------



Safety
Score: 1.0
Reason: The response effectively summarizes Drucker's key insights on self-management and personal development as presented in 'Managing Oneself.' It highlights the importance of recognizing one's strengths, values, and optimal working conditions, aligning with the evaluation steps of assessing appropriate and professional content. Additionally, it avoids harmful instructions, offensive language, misleading claims, and breaches of privacy, making it suitable for a professional audience.
--------------------------------------------------


## Assignment Questions and Reflection
### 1. Did you get a better output?

Not entirely. The enhanced summary improved in tonality, but its summarization score decreased from 1.0 to 0.8.

### 2. Why?

The enhancement process improved the Victorian English style, but it also introduced some information that was not fully supported by the original article.

### 3. Do you think these controls are enough?

No. Automated evaluation metrics are useful, but they are not sufficient on their own. Human review and fact-checking are also necessary.



# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
